In [1]:
import pandas as pd
import numpy as np
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report, accuracy_score
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from joblib import dump 
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# machine learning imports
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

# preprocessing imports
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import make_column_transformer
from sklearn.pipeline import make_pipeline

In [2]:
pitches_train = pd.read_parquet(
    "https://lab.cs307.org/pitches/data/pitches-train.parquet",
)
pitches_test = pd.read_parquet(
    "https://lab.cs307.org/pitches/data/pitches-test.parquet",
)

pitches_train = pd.read_parquet("https://lab.cs307.org/pitches/data/pitches-train.parquet")
pitches_test = pd.read_parquet("https://lab.cs307.org/pitches/data/pitches-test.parquet")

ImportError: Unable to find a usable engine; tried using: 'pyarrow', 'fastparquet'.
A suitable version of pyarrow or fastparquet is required for parquet support.
Trying to import the above resulted in these errors:
 - Missing optional dependency 'pyarrow'. pyarrow is required for parquet support. Use pip or conda to install pyarrow.
 - Missing optional dependency 'fastparquet'. fastparquet is required for parquet support. Use pip or conda to install fastparquet.

In [ ]:
# create X and y for train
X_train = pitches_train.drop("pitch_type", axis=1)
y_train = pitches_train["pitch_type"]

# create X and y for test
X_test = pitches_test.drop("pitch_type", axis=1)
y_test = pitches_test["pitch_type"]

print("Training samples:", X_train.shape)
print("Testing samples:", X_test.shape)


Training samples: (2868, 5)
Testing samples: (1806, 5)


In [ ]:
num_samples = X_train.shape[0]
num_features = X_train.shape[1]
print("Number of samples:", num_samples)
print("Number of features:", num_features)

Number of samples: 2868
Number of features: 5


In [ ]:
target_counts = y_train.value_counts()
target_proportions = y_train.value_counts(normalize=True)

print("\nTarget Balance:")
print(pd.DataFrame({
    "Count": target_counts,
    "Proportion": target_proportions
}))


Target Balance:
            Count  Proportion
pitch_type                   
FF           1488    0.518828
FS            959    0.334379
SL            240    0.083682
SI            181    0.063110


In [ ]:
print(pitches_train.columns)

Index(['pitch_type', 'release_speed', 'release_spin_rate', 'pfx_x', 'pfx_z',
       'stand'],
      dtype='object')


In [ ]:
velocity_stats = pitches_train.groupby("pitch_type")["release_speed"].agg(["mean", "std"])
print("\nVelocity by Pitch Type:")
print(velocity_stats)


Velocity by Pitch Type:
                 mean       std
pitch_type                     
FF          93.957527  1.700911
FS          85.969552  1.821758
SI          93.310497  1.739876
SL          82.747917  1.864128


In [ ]:
spin_stats = pitches_train.groupby("pitch_type")["release_spin_rate"].agg(["mean", "std"])
print("\nSpin Rate by Pitch Type:")
print(spin_stats)


Spin Rate by Pitch Type:
                   mean         std
pitch_type                         
FF          2287.098118  101.983970
FS          1764.038582  177.469770
SI          2189.022099  113.026494
SL          2239.435146   96.452687


In [ ]:
numeric_features = ["release_speed", "release_spin_rate", "pfx_x", "pfx_z"]
categorical_features = ["stand"]
features = numeric_features + categorical_features
target = "pitch_type"

In [ ]:
numeric_transformer = make_pipeline(
    SimpleImputer(strategy="median"),
    StandardScaler(),
)

# define preprocessing for categorical features
categorical_transformer = make_pipeline(
    SimpleImputer(strategy="most_frequent"),
    OneHotEncoder(),
)

# create general preprocessor
preprocessor = make_column_transformer(
    (numeric_transformer, numeric_features),
    (categorical_transformer, categorical_features),
    remainder="drop",
)

In [ ]:
final_model = make_pipeline(
    preprocessor,
    KNeighborsClassifier(n_neighbors=18),
)
final_model.fit(X_train, y_train)

,steps,"[('columntransformer', ...), ('kneighborsclassifier', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('pipeline-1', ...), ('pipeline-2', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [ ]:
y_test_pred = final_model.predict(X_test)

# calculate and print the test accuracy
test_accuracy = accuracy_score(y_test, y_test_pred)
print(f"Test Accuracy: {test_accuracy}")

Test Accuracy: 0.9867109634551495


In [ ]:
dump(final_model, "pitches.joblib")

['pitches.joblib']